In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf

In [ ]:
def get_event_tables_from_directory(csv_dir):
    """
    Processes all CSV files in a directory and generates a new CSV for each:
    - Extracts 'event number' and 'event time (ms)'
    - Saves to same directory as '<original_filename>_event_table.csv'
    """
    csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            if "event start time (ms)" not in df.columns:
                print(f"Skipping {csv_file} (missing 'event start time (ms)')")
                continue

            event_table = pd.DataFrame({
                "event number": range(len(df)),
                "event time (ms)": df["event start time (ms)"]
            })

            base_name = os.path.splitext(os.path.basename(csv_file))[0]
            save_path = os.path.join(csv_dir, f"{base_name}_event_table.csv")
            event_table.to_csv(save_path, index=False)
            print(f"Saved: {save_path}")

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

In [ ]:
tables = get_event_tables_from_directory("./data/ManualGranuleAnalysis_10_July_csvs")

# Access event table for a specific file (e.g., "25529004Events")
print(tables["25529004Events"].head())

In [ ]:
def extract_sweeps_by_event_numbers(abf_path, event_numbers):
    abf = pyabf.ABF(abf_path)
    waveforms = []

    for sweep in event_numbers:
        if sweep < abf.sweepCount:
            abf.setSweep(sweep)
            waveforms.append(abf.sweepY.copy())
        else:
            print(f"Skipping sweep {sweep} (not in {os.path.basename(abf_path)})")

    return np.array(waveforms), abf.sweepX

def average_waveforms_from_event_table(event_table, abf_path, time_windows):
    averaged_waveforms = []
    sweep_groups = []

    for start, end in time_windows:
        matching = event_table[
            (event_table["event time (ms)"] >= start) &
            (event_table["event time (ms)"] <= end)
        ]["event number"].astype(int).to_list()
        sweep_groups.append(matching)

    for group in sweep_groups:
        waveforms, time_axis = extract_sweeps_by_event_numbers(abf_path, group)
        if len(waveforms) > 0:
            mean_waveform = np.mean(waveforms, axis=0)
        else:
            mean_waveform = None
        averaged_waveforms.append(mean_waveform)

    return time_axis, averaged_waveforms

def plot_average_waveforms(time_axis, waveforms, labels, colors, title, save_path):
    plt.figure(figsize=(10, 6))
    for waveform, label, color in zip(waveforms, labels, colors):
        if waveform is not None:
            plt.plot(time_axis, waveform, label=label, color=color, lw=2)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (pA)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved: {save_path}")

def process_event_tables_and_abfs(csv_dir, abf_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    time_windows = [
        (0, 300000),
        (330000, 660000),
        (690000, 990000)
    ]
    labels = [
        "0–300,000 ms",
        "330,000–660,000 ms",
        "690,000–990,000 ms"
    ]
    colors = ["black", "red", "blue"]

    event_csvs = glob.glob(os.path.join(csv_dir, "*_event_table.csv"))

    for csv_file in event_csvs:
        try:
            df = pd.read_csv(csv_file)
            base_name = os.path.basename(csv_file).replace("_event_table.csv", "")
            abf_path = os.path.join(abf_dir, f"{base_name}.abf")

            if not os.path.exists(abf_path):
                print(f"No ABF file found for {base_name}")
                continue

            time_axis, averaged_waveforms = average_waveforms_from_event_table(df, abf_path, time_windows)

            save_path = os.path.join(output_dir, f"{base_name}_avg_waveforms.png")
            plot_average_waveforms(time_axis, averaged_waveforms, labels, colors, f"Average Waveforms: {base_name}", save_path)

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

In [ ]:
process_event_tables_and_abfs(
    csv_dir="./data/ManualGranuleAnalysis_10_July_csvs",
    abf_dir="./data/ManualGranuleEvents",
    output_dir="./data/ManualGranuleEvents"
)